# Run the Credit Score demo

## Apply the feature store definitions

Verify the client `feature_store.yaml`.

In [2]:
![ -f f43b44b.tar.gz ] || wget https://github.com/feast-dev/feast-credit-score-local-tutorial/archive/f43b44b.tar.gz
!kubectl cp f43b44b.tar.gz $(kubectl get pods -l 'feast.dev/name=example' -ojsonpath="{.items[*].metadata.name}"):/feast-data -c registry
!kubectl exec deploy/feast-example -c registry -- rm -rf /feast-data/feast-credit-score-local-tutorial
!kubectl exec deploy/feast-example -c registry -- mkdir /feast-data/feast-credit-score-local-tutorial
!kubectl exec deploy/feast-example -c registry -- tar vfx /feast-data/f43b44b.tar.gz -C /feast-data/feast-credit-score-local-tutorial --strip-components 1

feast-credit-score-local-tutorial-f43b44b245ae2632b582f14176392cfe31f98da9/.gitignore
feast-credit-score-local-tutorial-f43b44b245ae2632b582f14176392cfe31f98da9/LICENSE
feast-credit-score-local-tutorial-f43b44b245ae2632b582f14176392cfe31f98da9/README.md
feast-credit-score-local-tutorial-f43b44b245ae2632b582f14176392cfe31f98da9/app.py
feast-credit-score-local-tutorial-f43b44b245ae2632b582f14176392cfe31f98da9/credit_model.py
feast-credit-score-local-tutorial-f43b44b245ae2632b582f14176392cfe31f98da9/data/
feast-credit-score-local-tutorial-f43b44b245ae2632b582f14176392cfe31f98da9/data/credit_history.parquet
feast-credit-score-local-tutorial-f43b44b245ae2632b582f14176392cfe31f98da9/data/credit_history_sample.csv
feast-credit-score-local-tutorial-f43b44b245ae2632b582f14176392cfe31f98da9/data/loan_table.parquet
feast-credit-score-local-tutorial-f43b44b245ae2632b582f14176392cfe31f98da9/data/loan_table_sample.csv
feast-credit-score-local-tutorial-f43b44b245ae2632b582f14176392cfe31f98da9/data/tr

Move `feature_store.yaml` config to demo directory, then verify its contents.

In [3]:
!kubectl exec deploy/feast-example -c registry -- [ -f /feast-data/feature_store.yaml ] || kubectl exec deploy/feast-example -c registry -- mv /feast-data/credit_scoring_local/feature_repo/feature_store.yaml /feast-data/feature_store.yaml
!kubectl exec deploy/feast-example -c registry -- cp -f /feast-data/feature_store.yaml /feast-data/feast-credit-score-local-tutorial/feature_repo/feature_store.yaml
!kubectl exec deploy/feast-example -c registry -- cat /feast-data/feast-credit-score-local-tutorial/feature_repo/feature_store.yaml

command terminated with exit code 1
project: credit_scoring_local
provider: local
offline_store:
    type: duckdb
online_store:
    type: redis
    connection_string: redis.feast.svc.cluster.local:6379
registry:
    path: postgresql+psycopg://postgres@postgres.feast.svc.cluster.local:5432/feast
    registry_type: sql
    cache_ttl_seconds: 60
    sqlalchemy_config_kwargs:
        echo: false
        pool_pre_ping: true
auth:
    type: no_auth
entity_key_serialization_version: 3


Apply feature store data.

In [4]:
!kubectl exec deploy/feast-example -c registry -- feast -c /feast-data/feast-credit-score-local-tutorial/feature_repo apply

/usr/local/lib/python3.11/site-packages/feast/feature_store.py:575: RuntimeWarning: On demand feature view is an experimental feature. This API is stable, but the functionality does not scale well for offline retrieval
  warnings.warn(
No project found in the repository. Using project name credit_scoring_local defined in feature_store.yaml
Applying changes for project credit_scoring_local
Deploying infrastructure for zipcode_features
Deploying infrastructure for credit_history


In [5]:
!kubectl exec deploy/feast-example -c registry -- bash -c 'cd /feast-data/feast-credit-score-local-tutorial/feature_repo && feast materialize-incremental $(date -u +"%Y-%m-%dT%H:%M:%S")'

Materializing 2 feature views to 2025-01-09 01:51:04+00:00 into the redis online store.

zipcode_features from 2025-01-08 21:18:23+00:00 to 2025-01-09 01:51:04+00:00:
0it [00:00, ?it/s]
0it [00:00, ?it/s]
credit_history from 2025-01-08 21:18:23+00:00 to 2025-01-09 01:51:04+00:00:


## Execute feast commands inside the client Pod

Finally, we run the full test suite from the client folder.

In [6]:
!kubectl exec deploy/feast-example -c registry -- feast -c /feast-data/feast-credit-score-local-tutorial/feature_repo feature-views list
!kubectl exec deploy/feast-example -c registry -- feast -c /feast-data/feast-credit-score-local-tutorial/feature_repo on-demand-feature-views describe total_debt_calc
!kubectl exec deploy/feast-example -c registry -- feast -c /feast-data/feast-credit-score-local-tutorial/feature_repo entities list

NAME              ENTITIES     TYPE
zipcode_features  {'zipcode'}  FeatureView
credit_history    {'dob_ssn'}  FeatureView
total_debt_calc   {'dob_ssn'}  OnDemandFeatureView
spec:
  name: total_debt_calc
  features:
  - name: total_debt_due
    valueType: DOUBLE
  sources:
    application_data:
      requestDataSource:
        type: REQUEST_SOURCE
        requestDataOptions:
          schema:
          - name: loan_amnt
            valueType: INT64
        name: application_data
    credit_history:
      featureViewProjection:
        featureViewName: credit_history
        featureColumns:
        - name: credit_card_due
          valueType: INT64
        - name: mortgage_due
          valueType: INT64
        - name: student_loan_due
          valueType: INT64
        - name: vehicle_loan_due
          valueType: INT64
        - name: hard_pulls
          valueType: INT64
        - name: missed_payments_2y
          valueType: INT64
        - name: missed_payments_1y
          valueTyp

## Run test code inside the client Pod

Finally, we run the full test suite from the client folder.

In [ ]:
!kubectl exec deploy/feast-example -c registry -- python -m venv --system-site-packages /feast-data/venv
!kubectl exec deploy/feast-example -c registry -- bash -c 'source /feast-data/venv/bin/activate && pip install -r /feast-data/feast-credit-score-local-tutorial/requirements.txt'
!kubectl exec deploy/feast-example -c registry -- bash -c 'source /feast-data/venv/bin/activate && cd /feast-data/feast-credit-score-local-tutorial && python run.py'
!kubectl exec deploy/feast-example -c registry -- bash -c 'source /feast-data/venv/bin/activate && cd /feast-data/feast-credit-score-local-tutorial && streamlit run --server.port 8501 streamlit_app.py'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 16.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 24.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.2/540.2 kB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.2/731.2 kB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.2/326.2 kB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

```bash
$ kubectl port-forward deploy/feast-example 8501:8501
```
Then navigate to the local URL on which Streamlit is being served.

http://localhost:8501

```bash
$ kubectl port-forward svc/feast-example-online 8888:443
```
Then navigate to the local URL on which Streamlit is being served.

https://localhost:8888/docs#/